# 实验四：基于Auto-Encoder的图像压缩实验


## 实验介绍

降维是一种对高维度特征数据预处理方法。降维是将高维度的数据保留下最重要的一些特征，去除噪声和不重要的特征，从而实现提升数据处理速度的目的。在实际的生产和应用中，降维在一定的信息损失范围内，可以为我们节省大量的时间和成本。降维也成为应用非常广泛的数据预处理方法。本实验主要介绍使用MindSpore在图像数据集上使用PCA和Auto-Encoder模型进行图像压缩实验。

## 实验目的

- 了解降维算法和PCA的基本概念；
- 了解auto-encoder模型的用途与搭建方法

## 实验环境

- MindSpore 1.7.0

In [ ]:
%cd /home/ma-user/work

In [ ]:
!wget https://ascend-professional-construction-dataset.obs.myhuaweicloud.com/deep-learning/MNIST.zip
!unzip MNIST.zip

## 实验内容

### 实验原理

PCA（Principal Component Analysis）是一种使用最广泛的数据降维算法，常用于高维数据的降维，可用于提取数据的主要特征分量。其主要思想是将n维特征映射到k维上，这k维是全新的正交特征也被称为主成分，是在原有n维特征的基础上重新构造出来的k维特征。其算法步骤如下：

设有 m 条 n 维数据。

1. 将原始数据按列组成 n 行 m 列矩阵 X；
2. 将 X 的每一行进行零均值化，即减去这一行的均值；
3. 求出协方差矩阵 $C=\frac{1}{m}XX^T$ ；
4. 求出协方差矩阵的特征值及对应的特征向量；
5. 将特征向量按对应特征值大小从上到下按行排列成矩阵，取前 k 行组成矩阵 P；
6. $Y=PX$即为降维到 k 维后的数据。

自编码器（AutoEncoder）是一种用于无监督学习的神经网络模型，它的基本原理可以概括为以下几个步骤：

1. 编码器（Encoder）：这部分的网络将输入数据压缩成一个较低维度的表示形式（称为编码）。这个过程类似于数据压缩，目的是提取输入数据中最重要的特征
2. 编码（Code）：编码是编码器处理后的结果，它是输入数据的一个压缩表示。编码通常是一个远小于输入数据维度的向量，它捕捉了输入数据的核心特征
3. 解码器（Decoder）：解码器部分尝试从编码中重构原始输入数据。这个过程可以看作是对编码的解压缩。理想情况下，解码器的输出应该与原始输入数据尽可能相似
4. 损失函数（Loss Function）：自编码器的训练过程中，需要一个损失函数来度量解码器输出和原始输入之间的差异。常用的损失函数包括均方误差（MSE）等。通过最小化这个损失，网络学习到如何有效地编码和解码数据。

### 导入模块包并初始化环境

In [ ]:
import os
import numpy as np
import mindspore as ms
import mindspore.dataset as ds
import mindspore.dataset.transforms.c_transforms as C
import mindspore.dataset.vision.c_transforms as CV
import mindspore.ops as ops
import matplotlib.pyplot as plt
from easydict import EasyDict as edict
from mindspore import Tensor, Model, nn, context
from mindspore.train.serialization import load_checkpoint
from mindspore.train.callback import CheckpointConfig, ModelCheckpoint, LossMonitor

context.set_context(mode=context.PYNATIVE_MODE, device_target="GPU")

In [ ]:
cfg = edict({
    'channel': 1,  # 图片通道数
    'image_height': 32,  # 图片高度
    'image_width': 32,  # 图片宽度
    'batch_size': 128,
    'embed_size': 30,
    'data_dir': 'MNIST'
})

### 可视化

In [ ]:
def visualize(images, labels):
    plt.figure(figsize=(10, 3))
    for (i, img) in enumerate(images):
        if img.ndim == 2:
            img = np.expand_dims(img, axis=2)
        else:
            img = img.transpose(1, 2, 0)
        plt.subplot(1, 8, i+1)
        plt.imshow((img*255).astype(int), cmap='gray')
        plt.title('%s' % labels[i])
        plt.xticks([])
    plt.show()

### 构建数据集

In [ ]:
def create_dataset(data_dir=cfg.data_dir, training=True, batch_size=6, resize=(28, 28),
                   rescale=1/255, shift=0, buffer_size=64):
    data_train = os.path.join(data_dir, 'train') # 训练集信息
    data_test = os.path.join(data_dir, 'test') # 测试集信息
    ds = ms.dataset.MnistDataset(data_train if training else data_test)

    ds = ds.map(input_columns=["image"], operations=[CV.Resize(resize), CV.Rescale(rescale, shift), CV.HWC2CHW()])
    ds = ds.map(input_columns=["label"], operations=C.TypeCast(ms.int32))
    # When `dataset_sink_mode=True` on Ascend, append `ds = ds.repeat(num_epochs) to the end
    ds = ds.shuffle(buffer_size=buffer_size).batch(batch_size, drop_remainder=True)

    return ds

### PCA


In [ ]:
#数据中心化
def Z_centered(dataMat):
	rows,cols=dataMat.shape
	meanVal = np.mean(dataMat, axis=0)  # 按列求均值，即求各个特征的均值
	meanVal = np.tile(meanVal,(rows,1))
	newdata = dataMat-meanVal
	return newdata, meanVal
 
#协方差矩阵
def Cov(dataMat):
	rows = dataMat.shape[0]
	meanVal = np.mean(dataMat,0) #压缩行，返回1*cols矩阵，对各列求均值
	meanVal = np.tile(meanVal, (rows,1)) #返回rows行的均值矩阵
	Z = dataMat - meanVal
	Zcov = (1/(rows-1))*Z.T * Z
	return Zcov
	
#得到最大的k个特征值和特征向量
def EigDV(covMat, k):
	D, V = np.linalg.eig(covMat) # 得到特征值和特征向量
	eigenvalue = np.argsort(D)
	K_eigenValue = eigenvalue[-1:-(k+1):-1]
	K_eigenVector = V[:,K_eigenValue]
	return K_eigenValue, K_eigenVector
	
#得到降维后的数据
def getlowDataMat(DataMat, K_eigenVector):
	return DataMat * K_eigenVector
 
#重构数据
def Reconstruction(lowDataMat, K_eigenVector, meanVal):
	reconDataMat = lowDataMat * K_eigenVector.T + meanVal
	return reconDataMat
 
#PCA算法
def PCA(data, k):
	dataMat = np.float32(np.mat(data))
	#数据中心化
	dataMat, meanVal = Z_centered(dataMat)
	#计算协方差矩阵
	covMat = np.cov(dataMat, rowvar=0)
	#得到最大的k个特征值和特征向量
	_, V = EigDV(covMat, k)
	#得到降维后的数据
	lowDataMat = getlowDataMat(dataMat, V)
	#重构数据
	reconDataMat = Reconstruction(lowDataMat, V, meanVal)
	return lowDataMat, reconDataMat

In [ ]:
val_ds = create_dataset(training=False, resize=(cfg.image_height, cfg.image_width))
data = val_ds.create_dict_iterator().__next__()
oriImage = data['image']
labels = data['label']

images = oriImage.asnumpy().reshape(6, -1)

_, reconImages = PCA(images, cfg.embed_size)

reconImages = reconImages.A.astype(np.float64).reshape((6, 1, cfg.image_height, cfg.image_width))

# 可视化
visualize(oriImage.asnumpy(), labels.asnumpy())
visualize(reconImages, labels.asnumpy())

### Auto-Encoder

In [ ]:
# 定义CNN图像识别网络
class AutoEncoder_Net(nn.Cell):
    def __init__(self, in_channel, embeded_channel=2):
        super(AutoEncoder_Net, self).__init__()
        self.encoder = nn.SequentialCell([
            nn.Dense(in_channel, 400),
            nn.ReLU(),
            nn.Dropout(),
            nn.Dense(400, embeded_channel),
            nn.ReLU()
        ])
        self.decoder = nn.SequentialCell([
            nn.Dense(embeded_channel, 400),
            nn.ReLU(),
            nn.Dense(400, in_channel),
            nn.Sigmoid(),
        ])
        self.flatten = nn.Flatten()
    #构建模型
    def construct(self, x):
        x = self.flatten(x)
        lowData = self.encoder(x)
        reconData = self.decoder(lowData)
        return x, lowData, reconData

In [ ]:
# 定义CNN图像识别网络
class AutoEncoder_Net(nn.Cell):
    def __init__(self, in_channel, embeded_channel=2):
        super(AutoEncoder_Net, self).__init__()
        self.encoder = nn.SequentialCell([
            nn.Dense(in_channel, 512),
            nn.ReLU(),
            nn.Dropout(),
            nn.Dense(512, 128),
            nn.ReLU(),
            nn.Dropout(),
            nn.Dense(128, embeded_channel),
            nn.ReLU()
        ])
        self.decoder = nn.SequentialCell([
            nn.Dense(embeded_channel, 128),
            nn.ReLU(),
            nn.Dense(128, 512),
            nn.ReLU(),
            nn.Dense(512, in_channel),
            nn.Sigmoid(),
        ])
        self.flatten = nn.Flatten()
    #构建模型
    def construct(self, x):
        x = self.flatten(x)
        lowData = self.encoder(x)
        reconData = self.decoder(lowData)
        return x, lowData, reconData

In [ ]:
class vaeLoss(nn.loss.loss.LossBase):
    def __init__(self):
        super(vaeLoss, self).__init__()
        self.zeros = ops.ZerosLike()
        self.mse = nn.MSELoss(reduction='mean')

    def construct(self, data, label):
        oriData, _, reconData = data
        return self.mse(oriData, reconData)

In [ ]:
# 删除原始保存的模型参数
!rm -f ./exp/auto_encoder-5_468.ckpt

train_ds = create_dataset('MNIST', training=True, batch_size=cfg.batch_size, resize=(cfg.image_height, cfg.image_width))

net=AutoEncoder_Net(in_channel=cfg.image_height*cfg.image_width, embeded_channel=cfg.embed_size)
net_loss=vaeLoss()
optimizer=nn.Adam(net.trainable_params(), 0.001)

model = Model(net, loss_fn=net_loss, optimizer=optimizer)
loss_cb = LossMonitor(per_print_times=100)
config_ck = CheckpointConfig(save_checkpoint_steps=100,
                             keep_checkpoint_max=1)
ckpoint_cb = ModelCheckpoint(prefix='auto_encoder', directory='./exp/', config=config_ck)
print("============== Starting Training ==============")
model.train(5, train_ds, callbacks=[loss_cb, ckpoint_cb], dataset_sink_mode=False)
print("============== finish Training ==============")


### 验证

In [ ]:
CKPT = './exp/auto_encoder-5_468.ckpt'
eval_net = AutoEncoder_Net(in_channel=cfg.image_height*cfg.image_width, embeded_channel=cfg.embed_size)
load_checkpoint(CKPT, net=eval_net)
eval_net.set_train(False)

_, _, reconImg = eval_net(oriImage)

# 可视化原图
visualize(oriImage.asnumpy(), labels.asnumpy())

# 可视化压缩后复原
reconImg = reconImg.view(6, cfg.image_height, cfg.image_width)
visualize(reconImg.asnumpy(), labels.asnumpy())

## 实验结果与分析（15'）

### 任务一（5'）

实现PCA对图像的压缩，在实验报告上粘贴得到的部分图像结果

### 任务二（10'）

运行Auto-Encoder baseline，在实验报告上粘贴得到的部分图像结果，比较分别使用PCA和auto-encoder压缩并复原后的图像效果（Auto-encoder效果不一定比PCA好，有时间可以好好调调），进行一定的分析。
